# RAG 2
Multiple data source (PDF: Apple_form_10K.pdf, Excel: Sample_Financial_Data.xlsx)

In [80]:
from dotenv import load_dotenv
from pathlib import Path
import random
from pydantic import BaseModel
import json
import datetime
import yaml

import pymupdf4llm
import pandas as pd
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import openai
from rag_eval import RAGEvaluator
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
	 FaithfulnessMetric,
	 AnswerRelevancyMetric
)
import cohere
import bm25s
from typing import Literal
import numpy as np

In [3]:
load_dotenv(override=True)

True

In [4]:
# Define hyperparameters
CONFIG = {
    "experiment_name": "rag_2 + cohere_reranking + hybrid + metadata",
    "pdf_parser": "pymypdf4llm",
    "splitter": "MarkdownHeaderTextSplitter",
    "headers_to_split_on": [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")],
    "chunk_size": None,
    "chunk_overlap": None,
    "embedding_model": "text-embedding-3-large",
    "embedding_size": 3072,
    "vdb_top_k": 10,
    "bm25_k": 10,
    "cohere_k": 3,
}

In [75]:
Path.cwd()

PosixPath('/Users/manuyadav/Projects/rags_implementations/rag_2/notebook')

In [76]:
APPLE_DOC_PATH = list(Path.cwd().parent.rglob("Apple_form_10k.pdf"))
FINANCIAL_DATA_PATH = list(Path.cwd().parent.rglob("Sample_Financial_Data.xlsx"))

GD_PATH = str(Path.cwd() / "eval" / "golden_dataset.jsonl")
RETRIEVAL_TEST_RESULTS_PATH = (Path.cwd() / "eval" / "retrieval_test_results.jsonl")
RETRIEVAL_LOG_PATH = Path.cwd() / "eval" / "retrieval_logs.csv"
CLASSIC_RETRIEVAL_MATRICES_PATH = (Path.cwd() / "eval" / "classic_retrieval_logs.csv")
GENERATION_TEST_RESULTS_PATH = (Path.cwd() / "eval" / "generation_test_results.jsonl")
GENERATION_LOG_PATH = Path.cwd() / "eval" / "generation_logs.csv"

### Parsing and Chunking of PDF

> **NOTE:** As we want to add metadata to our chunks, so will be wrapping our chunks LangChain's **Document** class and add metadata like the file name and file type.

In [18]:
md_text = pymupdf4llm.to_markdown(str(APPLE_DOC_PATH[0]))

In [19]:
# Chunking
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# First chunking
text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
pdf_chunks = text_splitter.split_text(md_text)

In [20]:
# Since the splitter alreadt wraps the chunks in Document class, we can add more metadata to each chunk

pdf_chunk_ids = [f"pdf_chunk_{i}" for i in range(len(pdf_chunks))]

for index, chunks in enumerate(pdf_chunks):
    chunks.metadata["source"] = "apple_10k.md"
    chunks.metadata["doc_type"] = "PDF-Markdown"
    chunks.metadata["chunk_id"] = pdf_chunk_ids[index]

In [21]:
pdf_chunks[100].metadata

{'Header 1': 'For the fiscal year ended September 27, 2025',
 'Header 2': '_Income Taxes_',
 'source': 'apple_10k.md',
 'doc_type': 'PDF-Markdown',
 'chunk_id': 'pdf_chunk_100'}

### Parsing and Chunking of Excel 

In [22]:
financial_data = pd.read_excel(str(FINANCIAL_DATA_PATH[0]))

In [23]:
financial_data.head()

,Segment,Country,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Date,Month Number,Month Name,Year
0,Government,Canada,Carretera,NaN,1618.5,3,20,32370.0,0.0,32370.0,16185.0,16185.0,2014-01-01,1,January,2014
1,Government,Germany,Carretera,NaN,1321.0,3,20,26420.0,0.0,26420.0,13210.0,13210.0,2014-01-01,1,January,2014
2,Midmarket,France,Carretera,NaN,2178.0,3,15,32670.0,0.0,32670.0,21780.0,10890.0,2014-06-01,6,June,2014
3,Midmarket,Germany,Carretera,NaN,888.0,3,15,13320.0,0.0,13320.0,8880.0,4440.0,2014-06-01,6,June,2014
4,Midmarket,Mexico,Carretera,NaN,2470.0,3,15,37050.0,0.0,37050.0,24700.0,12350.0,2014-06-01,6,June,2014


In [24]:
# Row-as-a-string method
excel_chunks = []

excel_chunk_ids = [f"excel_chunk_{i}" for i in range(len(financial_data))]

for index, row in financial_data.iterrows():
    row_text = (
        f"In {row['Month Name']} {row['Year']}, the {row['Segment']} segment in {row['Country']} "
        f"purchased {row['Units Sold']:,} units of the product '{row['Product']}'. "
        f"The manufacturing price was ${row['Manufacturing Price']} and the sale price was ${row['Sale Price']}. "
        f"This transaction generated ${row['Gross Sales']:,} in gross sales with a discount band of ${row['Discount Band']} "
        f"amounting to ${row['Discounts']:,} in total discounts. Net sales were ${row[' Sales']}. "
        f"The Cost of Goods Sold (COGS) was ${row['COGS']:,}, resulting in a net profit of ${row['Profit']:,}."
    )

    doc = Document(
        page_content=row_text,
        metadata={
            "source": "financial_data.xlsx",
            "row_index": index,
            "doc_type": "Excel",
            "country": row["Country"],
            "product": row["Product"],
            "chunk_id": excel_chunk_ids[index],
        },
    )

    excel_chunks.append(doc)

In [25]:
excel_chunks[45].metadata

{'source': 'financial_data.xlsx',
 'row_index': 45,
 'doc_type': 'Excel',
 'country': 'France',
 'product': 'Amarilla',
 'chunk_id': 'excel_chunk_45'}

### Embedding in V DB

In [26]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=3072)

In [27]:
pdf_chunk_ids = [f"pdf_chunk_{i}" for i in range(len(pdf_chunks))]
excel_chunk_ids = [f"excel_chunk_{i}" for i in range(len(excel_chunks))]

In [28]:
db_name = Path.cwd() / "vdb"

if db_name.exists():
    Chroma(
        persist_directory=str(db_name), embedding_function=embeddings
    ).delete_collection()

vector_store = Chroma(
    collection_name="embeddings",
    embedding_function=embeddings,
    persist_directory=str(db_name),
)

vector_store.add_documents(documents=pdf_chunks, ids=pdf_chunk_ids)
vector_store.add_documents(documents=excel_chunks, ids=excel_chunk_ids)

print(f"Vectorstore created with {vector_store._collection.count()} documents")

Vectorstore created with 917 documents


### Metadata Filtering for V DB

In [29]:
class QueryIntentFilter(BaseModel):
    has_filter: bool
    file_type: Literal["PDF-Markdown", "Excel"] | None = None
    cleaned_query: (
        str  # The original query stripped of filter keywords for better vector matching
    )


def extract_query_intent(query: str) -> QueryIntentFilter:
    system_prompt = (
        "You are an AI assistant that analyzes user queries to extract metadata filters for a RAG system.\n"
        "Identify if the user is explicitly or implicitly asking for a specific file format ('PDF-Markdown' or 'Excel').\n"
        "Extract the filter and provide a cleaned version of the query stripped of file-type references."
    )

    completion = openai.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ],
        response_format=QueryIntentFilter,
        temperature=0.0,  # Keep it deterministic
    )

    return completion.choices[0].message.parsed

In [30]:
def retrieve_filtered_vector_docs(query: str, intent: QueryIntentFilter):

    search_filter = None
    if intent.has_filter and intent.file_type:
        search_filter = {"doc_type": intent.file_type}

    docs = vector_store.similarity_search(
        query=query, k=CONFIG["vdb_top_k"], filter=search_filter
    )

    return docs

### Hybrid Search

In [64]:
corpus_text = [i.page_content for i in pdf_chunks]
corpus_tokens = bm25s.tokenize(texts=corpus_text)
bm25s_retriever = bm25s.BM25(corpus=pdf_chunks)
bm25s_retriever.index(corpus_tokens)


def bm25_retriever(question, file_type_filter=None, k=10):
    question_tokens = bm25s.tokenize(texts=question)

    mask = None
    if file_type_filter:
        mask = np.array(
            [
                doc.metadata.get("file_type") == file_type_filter
                for doc in bm25s_retriever.corpus
            ]
        )

    results, _ = bm25s_retriever.retrieve(question_tokens, k=k, weight_mask=mask)

    return list(results[0])

DEBUG:bm25s:Building index from IDs objects           


In [33]:
def reciprocal_rank_fusion(
    vector_results: list, bm25_results: list, k: int = 60, top_n: int = 10
) -> list:
    rrf_scores = {}

    def score_results(doc_list):
        for rank, doc in enumerate(doc_list, start=1):
            # Extract the unique string ID to use as a hashable key
            chunk_id = doc.metadata["chunk_id"]

            if chunk_id not in rrf_scores:
                rrf_scores[chunk_id] = {"doc": doc, "score": 0.0}

            rrf_scores[chunk_id]["score"] += 1.0 / (k + rank)

    score_results(vector_results)
    score_results(bm25_results)

    sorted_items = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)

    return [item["doc"] for item in sorted_items[:top_n]]

### Reranking

In [34]:
co = cohere.ClientV2()


def cohere_reranking(question: str, retrieved_chunks: list) -> list:

    # Creating a YAML format, as we have metatdata as well (https://docs.cohere.com/docs/rerank-overview#example-with-structured-data)
    docs = [
        {
            "Header 1": i.metadata.get("Header 1"),
            "Header 2": i.metadata.get("Header 2"),
            "content": i.page_content,
        }
        for i in retrieved_chunks
    ]

    yaml_docs = [yaml.dump(doc, sort_keys=False) for doc in docs]

    results = co.rerank(
        model="rerank-v4.0-pro",
        query=question,
        documents=yaml_docs,
        top_n=CONFIG["cohere_k"],
    )

    return [retrieved_chunks[i.index] for i in results.results]

### Retrieval

In [35]:
def context_building(context: str) -> str:
    extended_prompt = f"""You are an smart AI assistant helps in reading form 10k and financial data. You will answer the question in explainatory way.
	If relevant, use the given context to answer any question.
	If you don't know the answer, say so.
	Context:
	{context}"""
    return extended_prompt

In [ ]:
def rag_pipeline(query: str) -> str:
    has_any_filter = extract_query_intent(query)
    cleaned_query = has_any_filter.cleaned_query
    file_filter_str = has_any_filter.file_type if has_any_filter.has_filter else None

    vdb_retrieved_docs = retrieve_filtered_vector_docs(cleaned_query, has_any_filter)
    bm25s_results = bm25_retriever(cleaned_query, file_type_filter=file_filter_str)
    hybrid_retrieval = reciprocal_rank_fusion(bm25s_results, vdb_retrieved_docs)

    reranking = cohere_reranking(cleaned_query, hybrid_retrieval)
    context = ["\n\n" + i.page_content for i in reranking]
    extended_prompt = context_building(context)
    result = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": extended_prompt},
            {"role": "user", "content": cleaned_query},
        ],
    )
    result = result.choices[0].message.content
    data = {
        "question": cleaned_query,
        "generated_answer": result,
        "contexts": [i.page_content for i in reranking],
        "chunk_used": [i.metadata["chunk_id"] for i in reranking],
    }
    print(data)
    return data

In [43]:
rag_pipeline("What were the total government sales in Canada for January?")

{'question': 'What were the total government sales in Canada for January?',
 'generated_answer': "In January 2014, the government segment in Canada had a transaction involving the product 'Carretera'. This transaction generated gross sales of $32,370.0, with net sales also amounting to $32,370.0 since there were no discounts applied. Therefore, the total government sales in Canada for January 2014 were $32,370.0.",
 'contexts': ["In January 2014, the Government segment in Canada purchased 1,618.5 units of the product 'Carretera'. The manufacturing price was $3 and the sale price was $20. This transaction generated $32,370.0 in gross sales with a discount band of $nan amounting to $0.0 in total discounts. Net sales were $32370.0. The Cost of Goods Sold (COGS) was $16,185.0, resulting in a net profit of $16,185.0.",
  "In December 2014, the Government segment in Canada purchased 2,852.0 units of the product 'Carretera'. The manufacturing price was $3 and the sale price was $350. This tra

## Evals

### Creating golden dataset 
Creating a synthetic golden dataset with multiple chunks  

In [18]:
class CoreFormat(BaseModel):
    question: str
    answer: str

In [19]:
gd_prompt = """
You are a QA engineer building a complex, multi-chunk evaluation dataset for a RAG system.
You will be provided with a list of text chunks retrieved from different sections or files.

Your task:
1. Review all the provided context chunks.
2. Synthesize the information to create ONE high-quality question that requires connecting the facts across at least 2 or 3 of these chunks to answer completely. 
3. Provide the comprehensive ground truth answer.

Output your response strictly as a JSON object with keys: "question" and "ground_truth_answer". No markdown wrappers.
"""

In [ ]:
def golden_dataset_generator(n: int) -> list:
    dataset = []

    # Clear the file before starting a fresh run to avoid duplicates
    if Path(GD_PATH).exists():
        Path(GD_PATH).unlink()

    for i in range(n):
        # Select random chunks (adjust to mix or single source as needed)
        random_chunks = random.choices(pdf_chunks, k=3)

        # Gather chunk ids and context in it
        chunk_ids = [chunk.metadata.get("chunk_id") for chunk in random_chunks]
        context_texts = [chunk.page_content for chunk in random_chunks]

        # Serialize context cleanly for the prompt
        serialized_context = ""
        for idx, text in enumerate(context_texts):
            serialized_context += f"--- Chunk {idx + 1} ---\n{text}\n\n"

        # Call OpenAI structured parsing
        result = openai.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": gd_prompt},
                {
                    "role": "user",
                    "content": f"Here are the retrieved chunks:\n{serialized_context}",
                },
            ],
            response_format=CoreFormat,
        )

        qa = result.choices[0].message.parsed

        gd_row = {
            "question": qa.question,
            "expected_output": qa.answer,
            "context": context_texts,
            "metadata": {
                "chunk_ids": chunk_ids,
                "sources": [chunk.metadata.get("source") for chunk in random_chunks],
            },
        }

        # Append row directly to JSONL file
        with open(GD_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(gd_row, ensure_ascii=False) + "\n")

        dataset.append(gd_row)

    return dataset

In [55]:
a = golden_dataset_generator(20)

### Logging 

In [39]:
def log_retrieval_deepeval(total_p, total_r, total_rel, count):

    log_entry = CONFIG.copy()
    log_entry["timestamp"] = datetime.datetime.now().strftime("%d/%b/%Y (%I:%M %p)")
    log_entry["ContextualPrecision"] = round(total_p / count, 3)
    log_entry["ContextualRecall"] = round(total_r / count, 3)
    log_entry["ContextualRelevancy"] = round(total_rel / count, 3)

    new_log = pd.DataFrame([log_entry])

    if RETRIEVAL_LOG_PATH.exists():
        logs_file = pd.read_csv(RETRIEVAL_LOG_PATH)
        logs_updated = pd.concat([logs_file, new_log])
        logs_updated.to_csv(RETRIEVAL_LOG_PATH, index=False)
    else:
        RETRIEVAL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        RETRIEVAL_LOG_PATH.touch()
        new_log.to_csv(RETRIEVAL_LOG_PATH, index=False)

    return "DeepEval's Retrieval Matrices Logged"

In [40]:
def log_classical():

    with open(RETRIEVAL_TEST_RESULTS_PATH, "r", encoding="utf-8") as f:
        items = [json.loads(line) for line in f if line.strip()]

    # Creating a rag_eval compatible list of dict (since column names are different, could be avoidable**)
    chunk_ids_test_cases = [
        {
            "question": i["question"],
            "retrieved_chunks": i["retrieved_chunk_ids"],
            "relevent_chunks": i["expected_chunk_ids"],
        }
        for i in items
    ]

    report = RAGEvaluator.from_dict_list(chunk_ids_test_cases, k=5)
    # RAGEvaluator(k=3).report(report, verbose=True)

    classic_retrieval_matrices_entry = CONFIG.copy()
    classic_retrieval_matrices_entry["timestamp"] = datetime.datetime.now().strftime(
        "%d/%b/%Y (%I:%M %p)"
    )
    classic_retrieval_matrices_entry["Hit@k"] = report.mean_hit_rate
    classic_retrieval_matrices_entry["MRR"] = report.mean_mrr
    classic_retrieval_matrices_entry["NDCG"] = report.mean_ndcg
    classic_retrieval_matrices_entry["Precision@k"] = report.mean_precision
    classic_retrieval_matrices_entry["Recall@k"] = report.mean_recall

    new_log = pd.DataFrame([classic_retrieval_matrices_entry])

    if CLASSIC_RETRIEVAL_MATRICES_PATH.exists():
        logs_file = pd.read_csv(CLASSIC_RETRIEVAL_MATRICES_PATH)
        logs_updated = pd.concat([logs_file, new_log])
        logs_updated.to_csv(CLASSIC_RETRIEVAL_MATRICES_PATH, index=False)
    else:
        CLASSIC_RETRIEVAL_MATRICES_PATH.parent.mkdir(parents=True, exist_ok=True)
        CLASSIC_RETRIEVAL_MATRICES_PATH.touch()
        new_log.to_csv(CLASSIC_RETRIEVAL_MATRICES_PATH, index=False)

    return "Matrices logged"

In [67]:
def log_generation_deepeval(mean_f, mean_ar):

    log_entry = CONFIG.copy()
    log_entry["timestamp"] = datetime.datetime.now().strftime("%d/%b/%Y (%I:%M %p)")
    log_entry["Faithfulness"] = round(mean_f, 3)
    log_entry["AnswerRelevancy"] = round(mean_ar, 3)

    new_log = pd.DataFrame([log_entry])

    if GENERATION_LOG_PATH.exists():
        logs_file = pd.read_csv(GENERATION_LOG_PATH)
        logs_updated = pd.concat([logs_file, new_log])
        logs_updated.to_csv(GENERATION_LOG_PATH, index=False)
    else:
        GENERATION_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        GENERATION_LOG_PATH.touch()
        new_log.to_csv(GENERATION_LOG_PATH, index=False)

    return "DeepEval's Generation Matrices Logged"

### Evaluation of Retrieval

Since it is a retrieval eval only, we'll be calling these 3 matrices only from **DeepEval**, 
1. **ContextualPrecisionMetric**
2. **ContextualRecallMetric**
3. **ContextualRelevancyMetric**

There's no need to call our `rag_pipeline()` to generate the answers. As the above matrices only require `question`, `expected_output` and `retrieved_context`.

In [68]:
def deepeval_retrieval_evaluator():

    # Initialize DeepEval retrieval metrics
    precision_metric = ContextualPrecisionMetric(threshold=0.7, model="gpt-4o-mini")
    recall_metric = ContextualRecallMetric(threshold=0.7, model="gpt-4o-mini")
    relevancy_metric = ContextualRelevancyMetric(threshold=0.7, model="gpt-4o-mini")

    retrieval_test_results = []

    total_p, total_r, total_rel = 0, 0, 0
    count = 0

    if RETRIEVAL_TEST_RESULTS_PATH.exists():
        RETRIEVAL_TEST_RESULTS_PATH.unlink()

    # Process your golden dataset
    with open(GD_PATH, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            question = row["question"]
            expected_ans = row["expected_output"]

            # Metadata filtering
            has_any_filter = extract_query_intent(question)
            cleaned_query = has_any_filter.cleaned_query
            file_filter_str = (
                has_any_filter.file_type if has_any_filter.has_filter else None
            )

            # Hybrid search with metadata filtering
            vdb_retrieved_docs = retrieve_filtered_vector_docs(
                cleaned_query, has_any_filter
            )
            bm25s_results = bm25_retriever(
                cleaned_query, file_type_filter=file_filter_str
            )
            hybrid_retrieval = reciprocal_rank_fusion(bm25s_results, vdb_retrieved_docs)

            # Reranking
            reranked_docs = cohere_reranking(cleaned_query, hybrid_retrieval)

            retrieved_contexts_text = [doc.page_content for doc in reranked_docs]
            retrieved_ids = [doc.metadata.get("chunk_id") for doc in reranked_docs]

            # Create the DeepEval TestCase
            test_case = LLMTestCase(
                input=question,
                expected_output=expected_ans,
                retrieval_context=retrieved_contexts_text,
            )

            precision_metric.measure(test_case)
            recall_metric.measure(test_case)
            relevancy_metric.measure(test_case)

            # Accumulate scores
            total_p += precision_metric.score
            total_r += recall_metric.score
            total_rel += relevancy_metric.score
            count += 1

            # Build your brief log entry in JSONL file
            result = {
                "timestamp": datetime.datetime.now().strftime("%d/%b/%Y (%I:%M %p)"),
                "question": question,
                "expected_chunk_ids": row["metadata"]["chunk_ids"],
                "retrieved_chunk_ids": retrieved_ids,
                "deepeval_precision": precision_metric.score,
                "deepeval_recall": recall_metric.score,
                "deepeval_relevancy": relevancy_metric.score,
                "precision_reason": precision_metric.reason,
                "recall_reason": recall_metric.reason,
            }

            retrieval_test_results.append(result)

            # Append result to file
            with open(str(RETRIEVAL_TEST_RESULTS_PATH), "a", encoding="utf-8") as out_f:
                out_f.write(json.dumps(result, ensure_ascii=False) + "\n")

    log_retrieval_deepeval(total_p, total_r, total_rel, count)

    log_classical()

    return "Retrieval eval cycle completed!"

In [ ]:
a = deepeval_retrieval_evaluator()

### Evaluation of Generation

In [81]:
def deepeval_generation_evaluator():

	faithfulness_metric = FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini")
	answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini")

	eval_results = []

	with open(GD_PATH, 'r', encoding='utf-8') as f:
		if RETRIEVAL_TEST_RESULTS_PATH.exists():
			RETRIEVAL_TEST_RESULTS_PATH.unlink()
		
		for line in f:
			row = json.loads(line)
			question = row['question']

			pipeline_output = rag_pipeline(question)
			
			test_case = LLMTestCase(
				input=question,
				actual_output=pipeline_output["generated_answer"],
				retrieval_context=pipeline_output["contexts"]
			)

			faithfulness_metric.measure(test_case)
			answer_relevancy_metric.measure(test_case)

			log_entry = {
				"question": question,
				"generated_answer": pipeline_output["generated_answer"],
				"faithfulness_score": faithfulness_metric.score,
				"answer_relevancy_score": answer_relevancy_metric.score,
				"faithfulness_reason": faithfulness_metric.reason,
				"relevancy_reason": answer_relevancy_metric.reason
			}
			eval_results.append(log_entry)

			# Append result to file
			with open(str(GENERATION_TEST_RESULTS_PATH), "a", encoding="utf-8") as out_f:
				out_f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")
			

	# Final average performance
	avg_faith = sum(item["faithfulness_score"] for item in eval_results) / len(eval_results)
	avg_rel = sum(item["answer_relevancy_score"] for item in eval_results) / len(eval_results)

	log_generation_deepeval(avg_faith, avg_rel)

	return "Generation eval cycle completed!"

In [ ]:
deepeval_generation_evaluator()